# 01 — Exploratory Data Analysis

Before introducing any missing values or applying imputation, we explore the two datasets:
- **California Housing** — purely numerical
- **Adult Census Income** — mixed (numerical + categorical)

Goals:
1. Understand the structure and distributions of each dataset
2. Identify correlations between features (relevant for MAR mechanism design)
3. Check for any naturally occurring missing values

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from sklearn.datasets import fetch_california_housing

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. California Housing

In [ ]:
housing = fetch_california_housing(as_frame=True)
df_housing = housing.frame
print(f'Shape: {df_housing.shape}')
df_housing.head()

In [ ]:
df_housing.info()

In [ ]:
df_housing.describe().round(2)

### 1.1 Distributions

In [ ]:
df_housing.hist(bins=40, figsize=(14, 10))
plt.suptitle('California Housing — Feature Distributions', y=1.02)
plt.tight_layout()
plt.show()

### 1.2 Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    df_housing.corr(),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    ax=ax
)
ax.set_title('California Housing — Correlation Matrix')
plt.tight_layout()
plt.show()

### 1.3 Missing Values

In [ ]:
missing = df_housing.isna().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values — dataset is complete.')

In [ ]:
msno.matrix(df_housing)
plt.title('California Housing — Missing Value Pattern')
plt.show()

## 2. Adult Census Income

In [ ]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
col_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]
df_adult = pd.read_csv(url, header=None, names=col_names, na_values=' ?', skipinitialspace=True)
print(f'Shape: {df_adult.shape}')
df_adult.head()

In [ ]:
df_adult.info()

In [ ]:
df_adult.describe().round(2)

### 2.1 Distributions — Numerical

In [ ]:
num_cols = df_adult.select_dtypes(include=np.number).columns
df_adult[num_cols].hist(bins=40, figsize=(14, 8))
plt.suptitle('Adult Census — Numerical Feature Distributions', y=1.02)
plt.tight_layout()
plt.show()

### 2.2 Distributions — Categorical

In [ ]:
cat_cols = df_adult.select_dtypes(exclude=np.number).columns.tolist()
fig, axes = plt.subplots(len(cat_cols), 1, figsize=(12, len(cat_cols) * 3))
for ax, col in zip(axes, cat_cols):
    df_adult[col].value_counts().plot(kind='bar', ax=ax)
    ax.set_title(col)
    ax.set_xlabel('')
plt.suptitle('Adult Census — Categorical Feature Distributions', y=1.01)
plt.tight_layout()
plt.show()

### 2.3 Correlation Matrix — Numerical

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    df_adult[num_cols].corr(),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    ax=ax
)
ax.set_title('Adult Census — Correlation Matrix (numerical features)')
plt.tight_layout()
plt.show()

### 2.4 Missing Values

In [ ]:
missing = df_adult.isna().sum()
missing = missing[missing > 0]
print('Naturally occurring missing values:')
print(missing)
print(f'\nTotal missing: {missing.sum()} ({missing.sum() / df_adult.size * 100:.2f}% of all values)')

In [ ]:
msno.matrix(df_adult)
plt.title('Adult Census — Missing Value Pattern')
plt.show()

## 3. Summary

Key observations to keep in mind for the imputation experiments:
- California Housing is complete — all missing values will be artificially introduced
- Adult Census has naturally occurring missing values in a few categorical columns
- Correlations observed above will guide the design of the MAR mechanism